In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

In [2]:
from extraction.agents import safe_generate,is_valid_json

In [3]:
from openai_agent import OpenAIAgent

In [4]:
import json

In [5]:
o_agent = OpenAIAgent('gpt-4o-mini',openai_key)
fallback_agent = OpenAIAgent('gpt-4o-mini','idk what')

In [6]:
with open('data/roles_data.json') as f:
    role_data = json.load(f)

In [11]:
fallback_skills(fake_resume,'Full Stack Developer')

{'skills': ['postgresql', 'rest', 'flask', 'python', 'aws', 'react']}

In [62]:
s = SkillExtractionAgent('gpt-4o-mini',openai_key,'Full Stack Developer')

In [63]:
s.run(fake_resume)

{'skills': ['Python', 'Java', 'C++', 'JavaScript', 'FastAPI', 'Flask', 'React', 'PyTorch', 'TensorFlow', 'Scikit-learn', 'PostgreSQL', 'MongoDB', 'Redis', 'Docker', 'Kubernetes', 'AWS', 'Machine Learning', 'System Design', 'REST APIs', 'Microservices', 'Data Structures & Algorithms', 'Pandas', 'Node.js', 'Socket.io']}


{'skills': ['Python',
  'Java',
  'C++',
  'JavaScript',
  'FastAPI',
  'Flask',
  'React',
  'PyTorch',
  'TensorFlow',
  'Scikit-learn',
  'PostgreSQL',
  'MongoDB',
  'Redis',
  'Docker',
  'Kubernetes',
  'AWS',
  'Machine Learning',
  'System Design',
  'REST APIs',
  'Microservices',
  'Data Structures & Algorithms',
  'Pandas',
  'Node.js',
  'Socket.io']}

In [64]:
class SkillExtractionAgent(OpenAIAgent):
    def __init__(self, model_slug, api_key, role):
        super().__init__(model_slug, api_key)
        self.role = role

        self.base_prompt = """You are an expert resume parser.

Extract ALL skills mentioned in the resume text below.

Instructions:
- Include technical skills, programming languages, frameworks, tools, libraries, and relevant concepts
- Do NOT include soft skills
- Do NOT hallucinate
- Normalize skills (e.g., "python" → "Python")
- Remove duplicates

Return ONLY valid JSON:
{{"skills": ["skill1", "skill2"]}}

Resume:
{resume_text}
"""

    def run(self, text):
        prompt = self.base_prompt.format(resume_text=text)
        output = self.generate_json(prompt,{"skills":[]})
        print(output)
        if not output:
            print('llm error')
            return fallback_skills(text, self.role)
        # try:
        #     parsed = json.loads(output)
        # except:
        #     print('json parse error')
        #     return fallback_skills(text, self.role)
        if "skills" not in output:
            print('wrong json schema')
            return fallback_skills(text, self.role)

        return output

def fallback_skills(text,role):
    with open('data/roles_data.json') as f:
        role_data = json.load(f)
    skills = set(role_data[role]['skills'])
    found = set()
    for word in text.split(' '):
        if word.lower() in skills:
            found.add(word.lower())
    return {'skills': list(found)}

In [70]:
class ExperienceExtractionAgent(OpenAIAgent):
    def __init__(self, model_slug, api_key):
        super().__init__(model_slug, api_key)

        self.base_prompt = """You are an expert resume parser.

Extract ALL work experience AND education details from the resume.

Instructions:
- Include job roles, companies, durations, responsibilities
- Include education (degree, university, year if present)
- Combine everything into a clean, structured paragraph
- Do NOT hallucinate
- Keep it concise but complete

Return ONLY valid JSON:
{{"experience": "full experience description here"}}

Resume:
{resume_text}
"""

    def run(self, text):
        curr_prompt = self.base_prompt.format(resume_text=text)

        response = safe_generate(self.client, 'Test')
        if not response['status'] == 'ok':
            return fallback_experience(text)

        out = super().generate_json(curr_prompt,{'experience':''})

        try:
            parsed = out
            if "experience" not in parsed or not parsed["experience"]:
                return fallback_experience(text)
            return parsed
        except:
            return fallback_experience(text)

def fallback_experience(text):
    text_lower = text.lower()

    lines = text.split("\n")

    exp_keywords = ["experience", "worked", "intern", "engineer", "developer"]
    edu_keywords = ["b.tech", "m.tech", "bachelor", "master", "phd", "university", "college"]

    extracted = []

    for line in lines:
        l = line.lower()

        if any(k in l for k in exp_keywords) or any(k in l for k in edu_keywords):
            extracted.append(line.strip())

    if not extracted:
        # fallback fallback (last safety)
        return {"experience": text[:500]}

    return {"experience": " ".join(extracted)}

In [71]:
e = ExperienceExtractionAgent('gpt-4o-mini',openai_key)

In [72]:
e.run(fake_resume)

{'experience': 'Software Engineer with 3+ years of experience in building scalable backend systems and data-driven applications. Strong background in machine learning, APIs, and cloud deployment. Interested in solving real-world problems using AI and distributed systems. Work Experience Software Engineer Junior Software Engineer Built backend services in Flask for internal analytics tools Built an NLP-based system to extract skills and experience from resumes B.Tech in Computer Science'}

In [76]:
class ProjectExtractionAgent(OpenAIAgent):
    def __init__(self, model_slug, api_key):
        super().__init__(model_slug, api_key)

        self.base_prompt = """You are an expert resume parser.

Extract ALL projects from the resume.

Instructions:
- Extract project names
- Include short description if available
- Do NOT hallucinate
- Keep entries concise

Return ONLY valid JSON:
{{"projects": ["project1", "project2"]}}

Resume:
{resume_text}
"""

    def run(self, text):
        curr_prompt = self.base_prompt.format(resume_text=text)

        response = safe_generate(self.client, 'Test')
        if not response['status'] == 'ok':
            return fallback_projects(text)

        out = super().generate_json(curr_prompt,{'projects':[]})

        try:
            parsed = out
            if "projects" not in parsed:
                return fallback_projects(text)
            return parsed
        except:
            return fallback_projects(text)

def fallback_projects(text):
    lines = text.split("\n")

    project_keywords = ["project", "built", "developed", "created", "designed"]

    projects = []

    for line in lines:
        l = line.lower()

        if any(k in l for k in project_keywords):
            projects.append(line.strip())

    # remove noise (very short lines)
    projects = [p for p in projects if len(p) > 10]

    if not projects:
        return {"projects": []}

    # deduplicate
    return {"projects": list(set(projects))}

In [77]:
p = ProjectExtractionAgent('gpt-4o-mini',openai_key)

In [78]:
p.run(fake_resume)

{'projects': ['Developed a scalable chat app with WebSockets',
  'Designed collaborative filtering-based recommendation system',
  'Built an NLP-based system to extract skills and experience from resumes',
  'Developed scalable REST APIs using FastAPI serving 50K+ daily users',
  'Designed microservices architecture improving system performance by 30%',
  'Built backend services in Flask for internal analytics tools']}

In [88]:
class ProfileExtractor():
    def __init__(self,model_slug,api_key,role):
        self.skill_extractor = SkillExtractionAgent(model_slug,api_key,role)
        self.project_extractor = ProjectExtractionAgent(model_slug,api_key)
        self.experience_extractor = ExperienceExtractionAgent(model_slug,api_key)
    def run(self,text):
        return {'skills': self.skill_extractor.run(text)['skills'],
        'projects': self.project_extractor.run(text)['projects'],
        'experience': self.experience_extractor.run(text)['experience']}

In [89]:
p = ProfileExtractor('gpt-4o-mini',openai_key,'Full Stack Developer')

In [90]:
p.run(fake_resume)

{'skills': ['Python', 'Java', 'C++', 'JavaScript', 'FastAPI', 'Flask', 'React', 'PyTorch', 'TensorFlow', 'Scikit-learn', 'PostgreSQL', 'MongoDB', 'Redis', 'Docker', 'Kubernetes', 'AWS', 'Machine Learning', 'System Design', 'REST APIs', 'Microservices', 'Data Structures & Algorithms', 'Pandas', 'Node.js', 'Socket.io']}


{'skills': ['Python',
  'Java',
  'C++',
  'JavaScript',
  'FastAPI',
  'Flask',
  'React',
  'PyTorch',
  'TensorFlow',
  'Scikit-learn',
  'PostgreSQL',
  'MongoDB',
  'Redis',
  'Docker',
  'Kubernetes',
  'AWS',
  'Machine Learning',
  'System Design',
  'REST APIs',
  'Microservices',
  'Data Structures & Algorithms',
  'Pandas',
  'Node.js',
  'Socket.io'],
 'projects': ['Developed a scalable chat app with WebSockets',
  'Designed collaborative filtering-based recommendation system',
  'Built an NLP-based system to extract skills and experience from resumes',
  'Developed scalable REST APIs using FastAPI serving 50K+ daily users',
  'Designed microservices architecture improving system performance by 30%',
  'Built backend services in Flask for internal analytics tools'],
 'experience': 'Software Engineer with 3+ years of experience in building scalable backend systems and data-driven applications. Strong background in machine learning, APIs, and cloud deployment. Interested in s

In [86]:
len(role_data)

6

In [6]:
len(role_data['Machine Learning Engineer'])

2

In [7]:
for x in role_data:
    print(len(role_data[x]))

2
2
2
2
2
2


In [8]:
fake_resume = """Name: Arjun Mehta
Email: arjun.mehta.dev@gmail.com

Phone: +91 98765 43210
Location: Bangalore, India
LinkedIn: linkedin.com/in/arjunmehta-dev
GitHub: github.com/arjunm-dev

Summary

Software Engineer with 3+ years of experience in building scalable backend systems and data-driven applications. Strong background in machine learning, APIs, and cloud deployment. Interested in solving real-world problems using AI and distributed systems.

Skills

Programming Languages: Python, Java, C++, JavaScript

Frameworks & Libraries: FastAPI, Flask, React, PyTorch, TensorFlow, Scikit-learn

Databases: PostgreSQL, MongoDB, Redis

Tools & Platforms: Docker, Kubernetes, AWS (EC2, S3, Lambda), Git

Concepts: Machine Learning, System Design, REST APIs, Microservices, Data Structures & Algorithms

Work Experience

Software Engineer
TechNova Solutions Pvt. Ltd. | July 2022 – Present

Developed scalable REST APIs using FastAPI serving 50K+ daily users

Designed microservices architecture improving system performance by 30%

Integrated ML models into production for recommendation systems

Optimized PostgreSQL queries reducing latency by 25%

Collaborated with frontend team using React for full-stack feature delivery

Junior Software Engineer
CodeBridge Labs | June 2020 – June 2022

Built backend services in Flask for internal analytics tools

Automated data pipelines using Python and Pandas

Implemented authentication systems (JWT, OAuth2)

Deployed applications on AWS EC2 with Docker

Projects

AI Resume Analyzer

Built an NLP-based system to extract skills and experience from resumes

Used SpaCy and BERT for entity recognition

Achieved 87% accuracy on custom dataset

Tech: Python, FastAPI, PyTorch

Real-Time Chat Application

Developed a scalable chat app with WebSockets

Supported 10K concurrent users

Implemented message queue using Redis

Tech: Node.js, Socket.io, Redis

E-Commerce Recommendation Engine

Designed collaborative filtering-based recommendation system

Increased user engagement by 18% in simulated environment

Tech: Python, Scikit-learn, Pandas

Education

B.Tech in Computer Science
Vellore Institute of Technology (VIT) | 2016 – 2020
CGPA: 8.7/10

Certifications

AWS Certified Solutions Architect – Associate

Deep Learning Specialization (Coursera)

Achievements

Ranked top 5% in company-wide coding competition

Solved 500+ problems on LeetCode"""

In [98]:
profile_json = {'skills': ['Python',
  'Java',
  'C++',
  'JavaScript',
  'FastAPI',
  'Flask',
  'React',
  'PyTorch',
  'TensorFlow',
  'Scikit-learn',
  'PostgreSQL',
  'MongoDB',
  'Redis',
  'Docker',
  'Kubernetes',
  'AWS',
  'Machine Learning',
  'System Design',
  'REST APIs',
  'Microservices',
  'Data Structures & Algorithms',
  'Pandas',
  'Node.js',
  'Socket.io'],
 'projects': ['Developed a scalable chat app with WebSockets',
  'Designed collaborative filtering-based recommendation system',
  'Built an NLP-based system to extract skills and experience from resumes',
  'Developed scalable REST APIs using FastAPI serving 50K+ daily users',
  'Designed microservices architecture improving system performance by 30%',
  'Built backend services in Flask for internal analytics tools'],
 'experience': 'Software Engineer with 3+ years of experience in building scalable backend systems and data-driven applications. Strong background in machine learning, APIs, and cloud deployment. Interested in solving real-world problems using AI and distributed systems. Work Experience Software Engineer Junior Software Engineer Built backend services in Flask for internal analytics tools Built an NLP-based system to extract skills and experience from resumes B.Tech in Computer Science'}

In [125]:
class GapIdentifierAgent(OpenAIAgent):
    def __init__(self, model_slug, api_key,role):
        super().__init__(model_slug, api_key)
        self.role = role
        self.base_prompt = """You are an expert skill gap analyzer.

Identify missing skills by comparing a candidate profile with multiple job descriptions.

Instructions:
- Extract all technical skills, tools, frameworks, and concepts from the job descriptions
- Ignore generic soft skills unless critical
- Normalize skills (lowercase, merge similar terms like "torch" → "pytorch")
- Compare with candidate profile skills
- Infer skills from projects and experience ONLY if clearly evident
- Do NOT mark equivalent skills as missing
- Do NOT hallucinate any skills

Categorization rules:
- core_missing → appears in ≥ 50 percent of job descriptions
- bonus_skills → not in core_missing

If candidate has equivalent skill (e.g., "tensorflow" vs "deep learning"), consider it covered.

Return ONLY valid JSON:
{{
  "core_missing": [...],
  "bonus_skills": [...]
}}

Input:
Profile:
{profile_json}

Job Descriptions:
{jd_list}"""

    def run(self,profile_json):
        with open('data/roles_data.json') as f:
            roles_data = json.load(f)
        jds = roles_data[self.role]['job_descriptions']
        curr_prompt = self.base_prompt.format(profile_json = profile_json,jd_list = jds)

        response = safe_generate(self, 'Test')
        print(response)
        if not response['status'] == 'ok':
            return fallback_gap_identification(profile_json,self.role)

        out = super().generate_json(curr_prompt,{'core_missing':[],'bonus_skills':[]})

        try:
            parsed = out
            if "core_missing" not in parsed or "bonus_skills" not in parsed:
                return fallback_gap_identification(profile_json,self.role)
            return parsed
        except:
            return fallback_gap_identification(profile_json,self.role)
import collections

def fallback_gap_identification(profile_json,role):
    with open('data/roles_data.json') as f:
        roles_data = json.load(f)
    jds = roles_data[role]['job_descriptions']
    skills = roles_data[role]['skills']
    skills_freq = [skill.lower() for skill in skills]
    for jd in jds:
        n = jd['preferred_skills'] + jd['tech_stack']
        n = [x.lower() for x in n]
        skills_freq.extend(n)
    skills_freq = collections.Counter(skills_freq)
    core_missing,bonus = [],[]
    for skill in skills_freq:
        if skills_freq[skill]>=(len(jds)//2):
            core_missing.append(skill)
        else:
            bonus.append(skill)
    core_missing,bonus = set(core_missing),set(bonus)
    for skill in profile_json['skills']:
        if skill.lower() in core_missing:
            core_missing.remove(skill.lower())
            #print(skill.lower())
        if skill.lower() in bonus:
            bonus.remove(skill.lower())
            #print(skill.lower())
    return {'core_missing':list(core_missing),'bonus_skills':list(bonus)}

In [126]:
g = GapIdentifierAgent('gpt-4o-mini',openai_key,'Full Stack Developer')

In [127]:
g.client

In [129]:
def safe_generate(agent, prompt):
    try:
        out = agent.generate(prompt)
        return {"status": "ok", "data": out}
    except Exception as e:
        return {
            "status": "api_error",
            "error": str(e)
        }

In [130]:
g.run(profile_json)

{'status': 'ok', 'data': "It looks like you're testing the system! How can I assist you today?"}


{'core_missing': ['typescript',
  'angular',
  'spring boot',
  'graphql',
  'ci cd pipelines',
  'cloud infrastructure',
  'authentication and authorization',
  'database design',
  'full stack development'],
 'bonus_skills': ['redux',
  'next.js',
  'firebase',
  'microservices architecture']}

In [108]:
fallback_gap_identification(profile_json,'Full Stack Developer')

python
java
javascript
flask
react
postgresql
mongodb
redis
docker
kubernetes
aws
node.js


{'core_missing': ['redux',
  'spring boot',
  'typescript',
  'graphql',
  'firebase',
  'microservices architecture',
  'next.js',
  'express'],
 'bonus_skills': ['responsive design',
  'nestjs',
  'cloud platforms',
  'startup experience',
  'logging and monitoring',
  'mvc architecture',
  'django',
  'sql',
  'full stack architecture',
  'angular',
  'git',
  'deployment pipelines',
  'scalable system design',
  'state management',
  'vue',
  'css',
  'client server architecture',
  'api security',
  'cloud deployment',
  'database design',
  'session management',
  'web application development',
  'jwt',
  'authentication flows',
  'backend development',
  'api design',
  'authentication and authorization',
  'restful api design',
  'web performance optimization',
  'frontend development',
  'rest',
  'database indexing',
  'caching strategies',
  'cross origin resource sharing',
  'nginx',
  'ci cd pipelines',
  'monolithic architecture',
  'system integration',
  'full stack dev

In [94]:
role_data['Full Stack Developer']['job_descriptions'][0]

{'title': 'Full Stack Developer - Web Applications',
 'summary': 'Develop end-to-end web applications including frontend interfaces, backend services, and database systems.',
 'responsibilities': ['build responsive user interfaces and backend services',
  'design and implement rest apis',
  'integrate frontend with backend systems',
  'manage databases and data models',
  'ensure performance scalability and security'],
 'required_skills': ['javascript or typescript',
  'react or angular or vue',
  'node.js or django or spring boot',
  'sql',
  'api design',
  'git'],
 'preferred_skills': ['docker', 'cloud platforms', 'graphql', 'redis'],
 'tech_stack': ['react', 'node.js', 'express', 'postgresql', 'docker', 'aws']}

In [131]:
from planning.agents import RoadMapAgent

ModuleNotFoundError: No module named 'planning'

In [6]:
from extraction.agents import ProfileBuilder
from openai_agent import OpenAIAgent

In [8]:
oai = OpenAIAgent('gpt-4o-mini',openai_key)
pb = ProfileBuilder('gpt-4o-mini',openai_key,role_data)

In [12]:
fallback_trigger = OpenAIAgent('gpt-4o-mini','wtf')

In [13]:
fallback_tester = ProfileBuilder(fallback_trigger,role_data)

In [13]:
pb.build_profile(fake_resume)

{'skills': {'core': ['Python', 'Java', 'C++', 'JavaScript'],
  'languages': ['Python', 'Java', 'C++', 'JavaScript'],
  'frameworks': ['FastAPI',
   'Flask',
   'React',
   'PyTorch',
   'TensorFlow',
   'Scikit-learn'],
  'tools': ['Docker',
   'Kubernetes',
   'AWS (EC2, S3, Lambda)',
   'Git',
   'Redis',
   'Node.js',
   'Socket.io'],
  'concepts': ['Machine Learning',
   'System Design',
   'REST APIs',
   'Microservices',
   'Data Structures & Algorithms']},
 'projects': [{'name': 'AI Resume Analyzer',
   'description': 'Built an NLP-based system to extract skills and experience from resumes using SpaCy and BERT for entity recognition. Achieved 87% accuracy on custom dataset.',
   'skills': ['Python', 'FastAPI', 'PyTorch'],
   'domain': 'Natural Language Processing',
   'complexity': 'Medium'},
  {'name': 'Real-Time Chat Application',
   'description': 'Developed a scalable chat app with WebSockets that supported 10K concurrent users and implemented a message queue using Redis.',


In [14]:
role_data['Full Stack Developer']['languages']

['javascript', 'typescript', 'python', 'java', 'sql']

In [17]:
for x in role_data['Full Stack Developer']:
    for y in x:
        y = set(y)
        for word in fake_resume.split(' '):
            if word.lower() in y:
                print(word)


a
a
a


In [33]:
words = fake_resume.split(' ')
for x in role_data['Full Stack Developer']:
    if x == 'job descriptions':
        continue
    for y in role_data['Full Stack Developer'][x]:
        #print(type(y))
        if isinstance(y, str):
            y = set(y.split(' '))
            for word in words:
                if word in y and word not in {'and','or','to'}:
                    print(word)
            #print(y)

frontend
backend
backend
architecture
authentication
system
system
feature
architecture
architecture
microservices
architecture
architecture
authentication
performance
scalable
scalable
system
system
scalable
pipelines


In [15]:
import re
text = fake_resume
role = 'Full Stack Developer'
ROLE_DATA = role_data
text = text.lower()

result = {
    "core": [],
    "languages": [],
    "frameworks": [],
    "tools": [],
    "concepts": []
}

if role not in ROLE_DATA:
    print(result)

role_data = ROLE_DATA[role]["skills"]

# 🔥 tokenize resume (clean words)
words = set(re.findall(r"\b\w+\b", text))

for category in result:
    for skill in role_data.get(category, []):
        if not isinstance(skill, str):
            continue

        skill = skill.lower()

        # 🔥 split skill into words
        skill_words = set(re.findall(r"\b\w+\b", skill))

        # 🔥 your idea: check if all skill words exist in resume
        if skill_words.issubset(words):
            result[category].append(skill)

# dedupe
for k in result:
    result[k] = list(set(result[k]))

print(result)

KeyError: 'skills'

In [16]:
import re

def fallback_skills(text, role, ROLE_DATA):
    text = text.lower()

    result = {
        "core": [],
        "languages": [],
        "frameworks": [],
        "tools": [],
        "concepts": []
    }

    if role not in ROLE_DATA:
        return result

    role_data = ROLE_DATA[role]

    # 🔥 tokenize resume
    words = set(re.findall(r"\b\w+\b", text))

    # 🔥 collect ALL skills (including JD)
    all_skills_by_cat = {k: set(role_data.get(k, [])) for k in result}

    for jd in role_data.get("job_descriptions", []):
        for key in ["required_skills", "preferred_skills", "tech_stack"]:
            for skill in jd.get(key, []):
                all_skills_by_cat.setdefault("tools", set()).add(skill)

    # 🔥 match skills
    for category, skills in all_skills_by_cat.items():
        for skill in skills:
            if not isinstance(skill, str):
                continue

            skill = skill.lower()
            skill_words = set(re.findall(r"\b\w+\b", skill))

            if skill_words and skill_words.issubset(words):
                result[category].append(skill)

    # dedupe
    for k in result:
        result[k] = list(set(result[k]))

    return result

In [17]:
fallback_skills(fake_resume,'Full Stack Developer',role_data)

{'core': ['full stack architecture'],
 'languages': ['javascript', 'python', 'java'],
 'frameworks': ['flask', 'node.js', 'react'],
 'tools': ['mongodb',
  'aws',
  'postgresql',
  'docker',
  'javascript',
  'redis',
  'node.js',
  'cloud platforms',
  'git',
  'react',
  'kubernetes',
  'cloud deployment',
  'java',
  'microservices architecture',
  'rest'],
 'concepts': ['scalable system design',
  'deployment pipelines',
  'jwt',
  'microservices architecture']}